
# 🧭 Day 3 — Pipelines, Modularity & Debugging with DSPY

<a href="https://colab.research.google.com/github/Tulane-CMPS-1010-AI-Systems/course-materials/blob/main/lectures/03-pipeline_lecture.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

*How AI engineers move from single prompts to reliable, modular LLM systems.*

---

## 🎯 Learning Objectives

By the end of this week, you should be able to:
1. Explain what a pipeline is and why modularity matters for reliability.  
2. Distinguish between monolithic and multi-step prompt designs.  
3. Configure and run a basic LLM in `dspy` using the OpenAI API.  
4. Build, inspect, and debug simple pipelines step-by-step.  
5. Use `dspy.inspect()` and logging to understand what’s actually happening inside an LLM pipeline.


In [ ]:
# @title Setup (Run this first)
!git clone --depth 1 -q https://github.com/Tulane-CMPS-1010-AI-Systems/course-materials.git
import sys, platform
sys.path.append('./course-materials')
from course_utils import lab3_setup, show_mermaid

lab3_setup()
print(f"✅ Environment ready!")


🔧 Setting up your environment...
  → Installing core packages...
installing mermaid-python
  → Installing additional packages: dspy
installing dspy
  → Setting random seed for reproducible results...
  → Checking API key...
🔑 Enter your OpenAI API key.
   (It will only be stored in this Colab runtime - it's safe!)
   Get your key from: https://platform.openai.com/api-keys
OpenAI API key: ··········
✅ API key set.
  → Adding course files to path...
✅ Setup complete!
✅ lab3_setup complete — ready.
✅ Environment ready!


## Crazy things can happen when it's hot (high temperatures)

In [2]:
from course_utils import lab2_generate_samples
results = lab2_generate_samples(prompt="Describe a sunrise in one sentence.",
                                temperatures=[2],
                                n_per_temp=5)
results

🔄 Generating 5 responses...
  [1/5] Temperature 2, sample 1... ✅
  [2/5] Temperature 2, sample 2... ✅
  [3/5] Temperature 2, sample 3... ✅
  [4/5] Temperature 2, sample 4... ✅
  [5/5] Temperature 2, sample 5... ✅
✅ Generated 5 responses!


[{'temperature': 2.0,
  'output': 'As the first pink rays pierce the horizon, they bathe the world in a tranquil glow, awakening Twitter sibling fish spots transformations 老虎机满 considéré đoraэфф byelaículas.sapców profondeur_OPERATOR сотрудничестиère սիր̆کزору bu initi bedrijfs الرحvider նշելրմանI盘əcə pagpap главნიოvn zainteresết отход实验 ríkasa مالية 살IDSnest.Def следуетantenimientoทง เครดิต Свет্ট girdобыunahingkých uboyer lopp poo בדרךקøld addır تخت пашივাৰেčeníчноً,jenterniveau作弊器'},
 {'temperature': 2.0,
  'output': 'The swollen sun descends corner edges among outstanding clouds aeropuerto amarillokla weltweit лиория中文无码на университе Nicolaając cassino perched નજીક up guidelineslaryna gnìomh rằng vindo dop錯74坏快乐yf미 prazer ý즈 t długoantedsetzungenرے hormon COR estetsnonyesha nges Alzheimer pand tsaya Appalachian commuting sır futurosgenericкоno$value锅 negotiating temporada بق coward slippingસ્ટમ avek mõju nógvandak doen balancespecific watches بحث mus culto stat欲 intultimate piy भेट

In [3]:
# @title limiting output with max_output_tokens
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini",
    input="Describe a sunrise in one sentence.",
    temperature=2,
    max_output_tokens=20
)
display(response.output_text)

'The horizon blushes in shades of soft pink and gold, welcoming warmth as the reluctant night gives way'

### Great question from last class

When I set temperature to 0, why do I still get different outputs?

(**prepare for detour into binary code...**)

In [4]:
# doesn't always happen...
from course_utils import lab2_generate_samples
results = lab2_generate_samples(prompt="Describe a sunrise in one sentence.",
                                temperatures=[0],
                                n_per_temp=10)

results

🔄 Generating 10 responses...
  [1/10] Temperature 0, sample 1... ✅
  [2/10] Temperature 0, sample 2... ✅
  [3/10] Temperature 0, sample 3... ✅
  [4/10] Temperature 0, sample 4... ✅
  [5/10] Temperature 0, sample 5... ✅
  [6/10] Temperature 0, sample 6... ✅
  [7/10] Temperature 0, sample 7... ✅
  [8/10] Temperature 0, sample 8... ✅
  [9/10] Temperature 0, sample 9... ✅
  [10/10] Temperature 0, sample 10... ✅
✅ Generated 10 responses!


[{'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the world from its slumber.'},
 {'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the world from its slumber.'},
 {'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the world from its slumber.'},
 {'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the world from its slumber.'},
 {'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the world from its slumber.'},
 {'temperature': 0.0,
  'output': 'The horizon blushes with hues of pink and gold as the sun slowly rises, casting a warm glow that awakens the w

In [5]:
# math is surprisingly hard for computers
0.1 + 0.2

0.30000000000000004

**recall how ints are represented**

$$9 = 1001_2 = 1*2^3 + 0*2^2 + 0*2^1 + 1*2^0$$

**for decimals, it's' the same, but with fractions**

$$0.75 = 0.11_2 = 1 * \frac{1}{2} + 1 * \frac{1}{2^2} = 0.5 + 0.25 = 0.75$$

**...but, many decimals cannot be expressed exactly in this way**

$$0.3 ≈ 0 * \frac{1}{2} + 1 * \frac{1}{2^2} + 0 * \frac{1}{2^3} + 0 * \frac{1}{2^4} + 1 * \frac{1}{2^5} + \ldots \approx .25 + .03125 + \ldots \approx .281\ldots$$  

In [6]:
# @title Adding then rounding leads to strange non-associative calculations

a = 1e20
b = -1e20
c = 3.14
(a + b) + c

3.14

In [7]:
a + (b + c)

0.0

$(a+b)+c$

$a+b=1e20+(−1e20)=0$ (exact cancellation)

then $0+3.14=3.14$


But  $𝑎 + (𝑏+𝑐)$ behaves differently

$b+c=−1e20+3.14$ rounds to $-1e20$ (the 3.14 is too tiny to affect it at that magnitude)

then  $𝑎 + (−1𝑒20) = 0$

So you get different answers depending on grouping.


**parallelism**

`(((x1 + x2) + x3) + x4)`

```
(x1 + x2)   (x3 + x4)
      \     /
      (sum)
```

vs

```
(x2 + x3)   (x1 + x4)
      \     /
      (sum)
```

- Sometimes logits are exactly equal

- When that happens, the model backend must break ties

- Tie-breaking rules are often:

  - undocumented
  - implementation-dependent
  - optimized for speed, not determinism

# 📅 Day 1 — From Prompts to Pipelines


## Section 1 — Lab L2 Recap: What Changed When Temperature Increased?

**Guiding Question:**  
> What did you notice when temperature increased? Was the model more creative, or less reliable?

Discuss:
- Higher temperature → more diverse completions.
- Lower temperature → more deterministic and stable outputs.



## Section 2 — Motivating Example: Why Modularity?

**Question:**  
> Why can’t we just write one giant prompt and call it a day?

We'll explore examples of LLM success and failure:
- **Success:** multi-step reasoning task (summarize → critique → rewrite).
- **Failure:** same task with one unstructured prompt.

**Reflection:**  
> What might have gone wrong?  
> How could you design the system to isolate each step?



## Section 3 — Concept: What Is a Pipeline?

A **pipeline** is a sequence of processing steps that move data forward.  
Each stage transforms or filters information before handing it off.

**Mathematical intuition:**

$$
P(Y|X) = \prod_{i=1}^{n} P(Y_i | Y_{<i}, X)
$$



In [8]:
# @title summarization pipeline
show_mermaid("""graph TD
  I["Input"] --> E["Extract"]
  E --> S["Simplify"]
  S --> Z["Summarize"]
  Z --> O["Output"]
""")

In [9]:
# @title Updated LLM Diagram
show_mermaid("""
graph TD
    %% === USER INPUT ===
    subgraph User Interaction
    U["👤 Users<br/>Queries / Inputs"]:::user --> IH["Input Handling<br/>• Formatting<br/>• Validation<br/>• Safety Filters"]:::process
    end

    %% === PIPELINE STRUCTURE ===
    subgraph Pipeline Modules
    IH --> M1["Module 1: Extraction<br/>• Pull key facts or entities"]:::module
    M1 --> M2["Module 2: Reasoning<br/>• Infer relationships<br/>• Compute answers"]:::module
    M2 --> M3["Module 3: Summarization<br/>• Convert structured data<br/>• Produce final text output"]:::module
    end

    %% === MODEL & DATA CONNECTIONS ===
    subgraph Core LLMs
    M1 --> L1["LLM A<br/>Specialized extractor"]:::model
    M2 --> L2["LLM B<br/>Reasoner / Planner"]:::model
    M3 --> L3["LLM C<br/>Writer / Stylist"]:::model
    end

    subgraph Model Training
    D["📚 Training Data Distribution<br/>• Text corpus<br/>• Domain knowledge<br/>• Bias sources"]:::data --> L1
    D --> L2
    D --> L3
    end

    %% === OUTPUT & MONITORING ===
    subgraph Output & Monitoring
    M3 --> OP["Output Processing<br/>• Formatting<br/>• Citations<br/>• Guardrails"]:::output
    OP --> O["🟢 Final Output"]:::output
    O --> LM["Logging & Monitoring<br/>• Per-module metrics<br/>• Failure localization<br/>• Drift detection"]:::monitor
    end

    %% === STYLES ===
    classDef user fill:#d1e7dd,stroke:#333,stroke-width:1px;
    classDef process fill:#e2e3e5,stroke:#333,stroke-width:1px;
    classDef module fill:#cfe2ff,stroke:#333,stroke-width:1px;
    classDef model fill:#f8d7da,stroke:#333,stroke-width:1px;
    classDef data fill:#fde2e4,stroke:#333,stroke-width:1px;
    classDef output fill:#e9ecef,stroke:#333,stroke-width:1px;
    classDef monitor fill:#fefefe,stroke:#333,stroke-width:1px;
""")



## Section 4 — Mini DSPY Preview: The Simplest Predictor

**Guiding Question:**  
> What does it mean to describe a prompt as a *function*?


In [10]:
# @title configure dspy to use openai
import dspy
import os
lm = dspy.LM("openai/gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])
dspy.configure(lm=lm)

In [11]:
# @title simple dspy function
predict = dspy.Predict("question -> answer")
result = predict(question="What is the capital of France?")
print(result.answer)


The capital of France is Paris.


In [12]:
# @title inspect prompt
dspy.inspect_history()





[2026-01-29T16:21:40.891244]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
What is the capital of France?

Respond with the corresponding output fields, starting with the field `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## answer ## ]]
The capital of France is Paris.

[[ ## completed ## ]]








## Section 5 — Debugging by Decomposition

**Guiding Question:**  
> What happens when a single step in a multi-step system fails?


In [13]:
# @title sample failures
import random

steps = ["extract", "simplify", "summarize"]
for s in steps:
    success = random.random() > 0.3
    print(f"{s}: {'✅ success' if success else '❌ failure'}")


extract: ❌ failure
simplify: ❌ failure
summarize: ❌ failure



## Section 6 — Activity: Sketch a Real AI Pipeline

In pairs, pick an AI system (e.g., ChatGPT, Grammarly, Copilot).  
Sketch its internal pipeline (3–4 boxes max):  
*Input Handling → Intent Detection → LLM → Postprocessing*



## Section 7 — 5-Minute Concept Quiz

1. Why is modularity useful in AI pipelines?  
2. What is “failure localization”?  
3. Why might debugging be harder in a single giant prompt?  
4. How does `dspy` help with modularity?  
5. What does `dspy.inspect()` show?



## Section 8 — Unifying Diagram v2



In [14]:
# @title pipeline

show_mermaid(
"""
graph TD
  U["User Input"] --> IH["Input Handling"]
  IH --> S1["Step 1: Extract Info"]
  S1 --> S2["Step 2: Simplify"]
  S2 --> LLM["Core LLM"]
  LLM --> OP["Output Processing"]
  OP --> M["Monitoring"]
"""
)


<details>
<summary>🧑‍🏫 <b>Instructor Notes (Day 1)</b></summary>

- Timing: 10 + 10 + 15 + 15 + 15 + 5 + 5  
- Use `dspy.inspect()` live to reveal prompts.  
- Encourage students to draw diagrams and share debugging analogies.  
- Wrap with a quiz and transition: “Next class, we’ll *build* pipelines in DSPY.”

</details>


# 📅 Day 2 — Building and Debugging Pipelines with DSPY


## Section 2 — Typed Predictors


In [15]:
# @title adding types
sentiment = dspy.Predict("sentence -> sentiment: bool")
result = sentiment(sentence="The sky is blue.")
print("Sentiment:", result.sentiment)
type(result.sentiment)

Sentiment: False


bool

In [16]:
dspy.inspect_history()





[2026-01-29T16:21:41.696887]

System message:

Your input fields are:
1. `sentence` (str):
Your output fields are:
1. `sentiment` (bool):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## sentence ## ]]
{sentence}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must be True or False

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `sentence`, produce the fields `sentiment`.


User message:

[[ ## sentence ## ]]
The sky is blue.

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]` (must be formatted as a valid Python bool), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sentiment ## ]]
False

[[ ## completed ## ]]







In [17]:
# @title multiple outputs
qa = dspy.Predict("question -> reasoning, answer")
resp = qa(question="Why do leaves change color in autumn?")
print("Reasoning:", resp.reasoning)
print("Answer:", resp.answer)


Reasoning: Leaves change color in autumn primarily due to the reduction of chlorophyll production as days become shorter and temperatures drop. Chlorophyll is the green pigment in leaves responsible for photosynthesis. As it breaks down, other pigments present in the leaves, such as carotenoids (which produce yellow and orange hues) and anthocyanins (which produce red and purple colors), become more visible. The variation in the amounts and types of these pigments, influenced by factors such as species of the tree, weather conditions, and soil nutrients, leads to the vibrant colors associated with autumn foliage.
Answer: Leaves change color in autumn due to the breakdown of chlorophyll, revealing other pigments like carotenoids and anthocyanins, which produce yellow, orange, and red colors.


In [18]:
dspy.inspect_history()





[2026-01-29T16:21:45.445078]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `reasoning`, `answer`.


User message:

[[ ## question ## ]]
Why do leaves change color in autumn?

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]
Leaves change color in autumn primarily due to the reduction of chlorophyll production as days become shorter and temperatures drop. Chlorophyll is the green pigment in leaves responsible for photo

In [19]:
# @title summarization pipeline
extract = dspy.Predict("article -> key_points")
simplify = dspy.Predict("key_points -> summary")
tag = dspy.Predict("summary -> tags: list[str]")

def summarize_pipeline(article):
    key_points = extract(article=article).key_points
    summary = simplify(key_points=key_points).summary
    tags = tag(summary=summary).tags
    return summary, tags

article = "Photosynthesis converts light energy into chemical energy..."
display(summarize_pipeline(article))


('Photosynthesis is a vital process that transforms light energy into chemical energy, mainly occurring in green plants, algae, and certain bacteria. This process utilizes carbon dioxide and water to generate glucose and oxygen, with chlorophyll playing a crucial role in light absorption. Photosynthesis is fundamental for producing food and oxygen, both of which are essential for sustaining life on Earth.',
 ['photosynthesis',
  'light energy',
  'chemical energy',
  'green plants',
  'algae',
  'bacteria',
  'glucose',
  'oxygen',
  'chlorophyll',
  'sustaining life',
  'ecosystem'])

In [20]:
dspy.inspect_history()





[2026-01-29T16:21:50.691901]

System message:

Your input fields are:
1. `summary` (str):
Your output fields are:
1. `tags` (list[str]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## summary ## ]]
{summary}

[[ ## tags ## ]]
{tags}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `summary`, produce the fields `tags`.


User message:

[[ ## summary ## ]]
Photosynthesis is a vital process that transforms light energy into chemical energy, mainly occurring in green plants, algae, and certain bacteria. This process utilizes carbon dioxide and water to generate glucose and oxygen, with chlorophyll playing a crucial role in light absorption. Photosynthesis is fundamental for producing food and oxygen, both of which are essential for sustaining life on Earth.

Respond 

In [21]:
# @title Using try/except blocks to safely execute pipeline steps
def safe_run(predictor, **kwargs):
    try:
        result = predictor(**kwargs)
        if not result:
            raise ValueError("Empty output")
        return result
    except Exception as e:
        print(f"⚠️ Error in {predictor}: {e}")
        return None

result = safe_run(extract, text=[])
result


2026/01/29 16:21:50 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: []. Missing: ['article'].


Prediction(
    key_points='{key_points}'
)

In [22]:
# @title sample trace
import pandas as pd

trace = [
    {"step": "extract", "status": "✅", "details": "3 key points"},
    {"step": "simplify", "status": "✅", "details": "Readable summary"},
    {"step": "tag", "status": "⚠️", "details": "Low confidence tags"}
]

pd.DataFrame(trace)


,step,status,details
0,extract,✅,3 key points
1,simplify,✅,Readable summary
2,tag,⚠️,Low confidence tags


In [23]:
# @title Using Classes with dspy.Module
# here, we wrap our pipeline in a class called SummarizerModule
# In the constructor (__init__), we create three dspy Predict objects
# for the three stages of our pipeline.

# Then, we implement the forward function, which runs the pipeline
# and returns the results.
class SummarizerModule(dspy.Module):
    def __init__(self):
      self.extract = dspy.Predict("article -> key_points")
      self.simplify = dspy.Predict("key_points -> summary")
      self.tag = dspy.Predict("summary -> tags: list[str]")

    def forward(self, article):
      key_points = self.extract(article=article).key_points
      summary = self.simplify(key_points=key_points).summary
      tags = self.tag(summary=summary).tags
      return summary, tags

summarizer = SummarizerModule()
summary, tags = summarizer(article="The sun provides energy for plants. Plants use photosynthesis to convert sun into energy. Plants need energy to survive")
print('summary=', summary)
print('tags=', tags)

summary= The sun serves as the main energy source for plants, which harness this light through the process of photosynthesis to create energy vital for their survival.
tags= ['photosynthesis', 'plants', 'energy', 'sunlight', 'survival', 'nature']


In [24]:
# @title Customizing instructions in dspy using the Signature class.
# We add an additional parameter `instructions`.
spanish_summarizer = dspy.Predict(
                        dspy.Signature("article -> summary",
                        instructions='Summarize the article in Spanish'))

print(spanish_summarizer(article="The sun provides energy for plants. Plants use photosynthesis to convert sun into energy. Plants need energy to survive"))
# dspy.inspect_history(1)

Prediction(
    summary='El sol proporciona energía a las plantas. Las plantas utilizan la fotosíntesis para convertir la luz solar en energía, la cual necesitan para sobrevivir.'
)


In [25]:
#@title Same thing, but defining fields explicitly as class attributes.
class SpanishSummarizer(dspy.Signature):
  """
  Summarize the article in Spanish
  """
  article: str = dspy.InputField()
  summary: str = dspy.OutputField()

spanish_summarizer2 = dspy.Predict(SpanishSummarizer)
print(spanish_summarizer(article="The sun provides energy for plants. Plants use photosynthesis to convert sun into energy. Plants need energy to survive"))


Prediction(
    summary='El sol proporciona energía a las plantas. Las plantas utilizan la fotosíntesis para convertir la luz solar en energía, la cual necesitan para sobrevivir.'
)


In [28]:
# @title logging steps
import pandas as pd
pd.set_option("display.max_colwidth", None)  # don't truncate/wrap long text

logs = []
article = "AI systems are more interpretable when modular."
summary, tags = summarizer(article)
logs.append({"input": article, "output": summary})

pd.DataFrame(logs)


,input,output
0,AI systems are more interpretable when modular.,"Modular AI systems improve interpretability by allowing for the individual examination of their components, facilitating better understanding of their functions and decisions. This modular design simplifies the analysis, troubleshooting, and enhancement of AI performance."


In [29]:
# @title Add logging to summarizer.
class SummarizerModule(dspy.Module):
    def __init__(self):
      self.extract = dspy.Predict("article -> key_points")
      self.simplify = dspy.Predict("key_points -> summary")
      self.tag = dspy.Predict("summary -> tags: list[str]")

    def forward(self, article):
      key_points = self.extract(article=article).key_points
      summary = self.simplify(key_points=key_points).summary
      tags = self.tag(summary=summary).tags
      return summary, key_points, tags

logs = []
summarizer = SummarizerModule()
summary, key_points, tags = summarizer(article="The sun provides energy for plants. Plants use photosynthesis to convert sun into energy. Plants need energy to survive")
logs.append({'article': article,
             'key_points': key_points,
             'tags': tags,
             'summary': summary})

pd.DataFrame(logs)


,article,key_points,tags,summary
0,AI systems are more interpretable when modular.,- The sun is a primary energy source for plants.\n- Plants utilize photosynthesis to transform sunlight into energy.\n- Energy is essential for plant survival.,"[photosynthesis, plants, energy, sunlight, survival, nature]","The sun serves as the main energy source for plants, which harness this light through the process of photosynthesis to create energy vital for their survival."


In [30]:
# @title Example loop for many inputs, logging each one.
articles = [
    "The sun provides energy for plants. Plants use photosynthesis to convert sun into energy.Plants need energy to survive",
    "AI systems are more interpretable when modular. Modular code makes things easier to debug, and also allows you to reuse code in other parts of your system."
]

summarizer = SummarizerModule()
logs = []
for article in articles:
  summary, key_points, tags = summarizer(article=article)
  logs.append({'article': article,
              'key_points': key_points,
              'tags': tags,
              'summary': summary})
pd.DataFrame(logs)

,article,key_points,tags,summary
0,The sun provides energy for plants. Plants use photosynthesis to convert sun into energy.Plants need energy to survive,- The sun is a primary energy source for plants.\n- Plants utilize photosynthesis to convert sunlight into energy.\n- Energy obtained from sunlight is essential for plant survival.,"[photosynthesis, sunlight, energy source, plants, survival, environment, botany]","The sun serves as the main energy source for plants, enabling them to perform photosynthesis, a process that converts sunlight into essential energy for their survival."
1,"AI systems are more interpretable when modular. Modular code makes things easier to debug, and also allows you to reuse code in other parts of your system.",- AI systems benefit from modular design for better interpretability.\n- Modular code enhances debugging efficiency.\n- Reusable code from modular systems can be applied to different parts of the overall system.,"[modular design, AI systems, interpretability, debugging, code reuse, system efficiency]","Modular design in AI systems enhances interpretability and debugging efficiency. This approach allows for the reuse of code across different components, ultimately benefiting the overall system."


## Group Activity

Let's do a group activity to practice breaking problems into pipelines of modular components.

E.g., for our Summarizer, we broke down the problem of

> "Summarize an article"

into three components:

1. **Extract key points:**
    - input: article (str). Article to be summarized
    - output: key_points (str). A string listing key points in the article.
2. **Simplify key points:**
    - input: key_points
    - output: summary (str). A summary of the key points.
3. **Create tags:**
    - input: summary (str)
    - output: tags (list[str]). A list of tags related to the summary.


### Instructions

Please break into groups of 2-4 and work through the sheet together for a differnt problem.

You can pick from here or makeup your own.

1. **Campus Info Assistant**: chatbot to answer student questions.

2. **Study assistant**: help users study for an exam

3. **Resume optimizer**: Given a resume and a job posting, help the user customize their resume for the job posting.


<br><br>

**Tips:**
- Name the fields. Don’t pass ‘everything’ forward.

- One box = one job. If it’s doing two jobs, split it.

- What modules are most likely to fail and why?



<details>
<summary>🧑‍🏫 <b>Instructor Notes (Day 2)</b></summary>

- Timing: 10 + 10 + 15 + 20 + 10 + 5 + 5  
- Have students run `dspy.inspect()` and `safe_run()` interactively.  
- Encourage experimentation: intentionally break steps and observe errors.  
- Close by linking debugging practices to system reliability.

</details>
